In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pad
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from prophet import Prophet
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

### Downloading the data

In [ ]:
file_path = "city_temperature.csv"
df_temperature = pad.read_csv(file_path)

file_path = "co2_conc.csv"
df_CO2_emission = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "arunavsutar/daily-atmosphere-carbon-dioxide-concentration",
        file_path,)

file_path = "sealevel.csv"
df_sea_level = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "kkhandekar/global-sea-level-1993-2021",
        file_path,)

### Processing the data and analyse it

In [ ]:
df_sea_level.drop(columns = ["TotalWeightedObservations","GMSL_noGIA","StdDevGMSL_noGIA","GMSL_GIA","StdDevGMSL_GIA","SmoothedGSML_GIA","SmoothedGSML_noGIA"])
df_sea_level = df_sea_level[(df_sea_level['Year'] >= 2013) & (df_sea_level['Year'] <= 2020)]
df_sea_level = df_sea_level.groupby(['Year']).mean().reset_index()

In [ ]:
df_temperature = df_temperature.drop(columns = ["Region","Country","State"])
df_temperature = df_temperature[(df_temperature['Year'] >= 2013) & (df_temperature['Year'] <= 2020)]
df_temperature  = df_temperature [df_temperature ['City'] == 'Paris']
df_temperature.rename(columns={'AvgTemperature': 'avg_city_temp'}, inplace=True)

In [ ]:
df_CO2_emission.drop(columns = ["Unnamed: 0","cycle"], inplace=True)
df_CO2_emission = df_CO2_emission[(df_CO2_emission['year'] >= 2013) & (df_CO2_emission['year'] <= 2020)]
df_CO2_emission.rename(columns={'year': 'Year'}, inplace=True)
df_CO2_emission.rename(columns={'month': 'Month'}, inplace=True)
df_CO2_emission.rename(columns={'day': 'Day'}, inplace=True)
df_CO2_emission.rename(columns={'trend': 'concentration_in_CO2'}, inplace=True)

In [ ]:
df_merged = df_temperature.merge(df_CO2_emission, on=['Year', 'Month', 'Day'], how='inner').merge(df_sea_level, on=['Year'], how='inner')

In [ ]:
def Data_information(name_df):

        if name_df == "sea_level":
            df_chosen= df_sea_level
        elif name_df == "temperature":
            df_chosen=df_temperature
        elif name_df == "CO2":
            df_chosen = df_CO2_emission
        elif name_df == "merge":
            df_chosen = df_merged
        
        print(f'EDA on the data {name_df}:')
        print("First 5 records:\n", df_chosen.head(),"\n")
        print("Describe:\n",df_chosen.describe(),"\n")
        print("Shape\n", df_chosen.shape,"\n")
        print("Information\n")
        df_chosen.info() #print directly doesn't return anything
        print("Null values:\n", df_chosen.isnull().sum(),"\n")
        profile = ProfileReport(df_chosen, title=f"Profiling Report on the Pima Indians Diabetes dataset_{name_df}")
        profile.to_notebook_iframe()
        print("\n----------------------------------------------------------------------------------------------------------------------------------\n")
        input("Press enter to continue")
        print("\n\n\n\n\n")

In [ ]:
Data_information("sea_level")
Data_information("temperature")
Data_information("CO2")

In [ ]:
df_paris = df_merged[df_merged['City'] == 'Paris']
corr_matrix = df_paris[["concentration_in_CO2","avg_city_temp","SmoothedGSML_GIA_sigremoved"]].corr()
plt.figure(figsize=(10, 7))
ax = sns.heatmap(corr_matrix, annot=True)
plt.title("Matrice de corrélation")
plt.show()

In [ ]:
def predict_sea_level(df):
    # Filter data for one city (here Paris)
    df_paris = df[df["City"] == "Paris"].copy()

    # Select features and target
    features = ["concentration_in_CO2"]
    target = "SmoothedGSML_GIA_sigremoved"

    X = df_paris[features]
    y = df_paris[target]

    # Train model
    model = LinearRegression()
    model.fit(X, y)

    # Predict
    y_pred = model.predict(X)

    # Evaluation
    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    print("R² score:", r2_score(y, y_pred))
    print("RMSE:", rmse)
    print("MAE:", mae)

    # Visualization
    plt.figure(figsize=(10, 5))
    plt.plot(df_paris["ds"], y, label="Real Sea level", alpha=0.6)
    plt.plot(df_paris["ds"], y_pred, color='red', label="Predicted Sea level")
    plt.xlabel("Date")
    plt.ylabel("Sea level based on the CO2 concentration")
    plt.title("Predicted vs Real Sea Level predicted based on the CO2 concentration- Linear Regression")
    plt.legend()
    plt.show()

# Prepare your datetime column (if not already done)
df_merged["ds"] = pad.to_datetime(df_merged[['Year', 'Month', 'Day']])

# Call the function
predict_sea_level(df_merged)

As you can see in the above code, We were not able to split the data in train and test since that for the representations, We wanted to be able to see the predictions over the time. Of course this is not optimal but we also did the same linear regression, but this time splitting the data into train and test :

In [ ]:
def predict_sea_level_split(df):
    # Filter data for one city (e.g., Paris)
    df_paris = df[df["City"] == "Paris"].copy()

    # Drop rows with missing or invalid temperature

    # Select features and target
    features = ["concentration_in_CO2"]
    target = "SmoothedGSML_GIA_sigremoved"

    X = df_paris[features]
    y = df_paris[target]
    dates = df_paris["ds"]

    X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(X, y, dates, test_size=0.2, random_state=42, shuffle=False)

    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print("RMSE:", rmse)
    print("MAE:", mae)

    # Visualization
    plt.figure(figsize=(12, 6))
    plt.plot(dates_test, y_test, label="Real Sea level", alpha=0.6)
    plt.plot(dates_test, y_pred, color='red', label="Predicted Sea level")
    plt.xlabel("Date")
    plt.ylabel("Sea level based on the CO2 concentration")
    plt.title("Predicted vs Real Sea Level predicted based on the CO2 concentration- Linear Regression")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Prepare your datetime column (if not already done)
df_merged["ds"] = pad.to_datetime(df_merged[['Year', 'Month', 'Day']])

# Call the function
predict_sea_level_split(df_merged)

Now that we splitted the data, even though the visualization over the time is not the best, we have better result since the RMSE and MAE are lower.

### Testing RandomForestRegressor

In [ ]:


# Define the model, here RandomForestRegressor
regr = RandomForestRegressor(max_depth=2, random_state=0)

# Define features and target
features = ["concentration_in_CO2", "SmoothedGSML_GIA_sigremoved"]
target = "avg_city_temp"

# Drop the wrong values
df_paris = df_paris[df_paris["avg_city_temp"] != -99]

X = df_paris[features]
y = df_paris[target]

# Separating in train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict
y_pred = rf_model.predict(X_test)

# Evaluating the model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("R² score:", r2_score(y_test, y_pred))
print("RMSE:", rmse)
print("MAE:", mae)

# 6. Visualisation
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_test, y=y_pred)
plt.xlabel("Température réelle")
plt.ylabel("Température prédite")
plt.title("Random Forest: prédiction vs réalité")
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')  # ligne diagonale parfaite
plt.show()

# 7. Importance des variables
importances = rf_model.feature_importances_
for name, score in zip(features, importances):
    print(f"{name} : {score:.3f}")

In this part, we tried to first predict the future sea level and CO2 concentration, to than use a random forest, predict the temperature and visualize the results using the future sea level and CO2 concentration.

We tried to predict the sea level and concentration in CO2 without using the temporal data, but as we can see below, the results are really not what we should expect.

In [ ]:
# Train CO2 model based on the year, but with a LinearRegression it is not a good way
co2_model = LinearRegression()
co2_model.fit(df_paris[["SmoothedGSML_GIA_sigremoved"]], df_paris["concentration_in_CO2"])

# Train sea level model based on the year, but with a LinearRegression it is not a good way
sea_model = LinearRegression()
sea_model.fit(df_paris[["concentration_in_CO2"]], df_paris["SmoothedGSML_GIA_sigremoved"])

# Define future years
future_years = np.arange(2025, 2051).reshape(-1, 1)

# Predict CO2 and sea level
future_co2 = co2_model.predict(future_years)
future_sea = sea_model.predict(future_years)

# Create prediction DataFrame
future_df = pad.DataFrame({
    "Year": future_years.flatten(),
    "concentration_in_CO2": future_co2,
    "SmoothedGSML_GIA_sigremoved": future_sea
})

future_df

And when we predicted it with the years, it was better so we will keep it.

In [ ]:
# Train CO2 model based on the year, but with a LinearRegression it is not a good way
co2_model = LinearRegression()
co2_model.fit(df_paris[["Year"]], df_paris["concentration_in_CO2"])

# Train sea level model based on the year, but with a LinearRegression it is not a good way
sea_model = LinearRegression()
sea_model.fit(df_paris[["Year"]], df_paris["SmoothedGSML_GIA_sigremoved"])

# Define future years
future_years = np.arange(2025, 2051).reshape(-1, 1)

# Predict CO2 and sea level
future_co2 = co2_model.predict(future_years)
future_sea = sea_model.predict(future_years)

# Create prediction DataFrame
future_df = pad.DataFrame({
    "Year": future_years.flatten(),
    "concentration_in_CO2": future_co2,
    "SmoothedGSML_GIA_sigremoved": future_sea
})

future_df

In [ ]:
# Variables explicatives = CO2 et sea level uniquement
X = df_paris[["concentration_in_CO2", "SmoothedGSML_GIA_sigremoved"]]
y = df_paris["avg_city_temp"]

rf_model = RandomForestRegressor(max_depth=4, random_state=0)
rf_model.fit(X, y)

# Predict temperature with your previously trained Random Forest model
future_df["predicted_temp"] = rf_model.predict(future_df[["concentration_in_CO2", "SmoothedGSML_GIA_sigremoved"]])

# Plot
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(future_df["Year"], future_df["predicted_temp"], marker="o", color="darkorange")
plt.xlabel("Year")
plt.ylabel("Predicted Avg City Temp (°C)")
plt.title("Predicted Temperature Trend in Paris (2025–2050)")
plt.grid(True)
plt.show()


Unfotunately, as we can see on the visualization, the RandomForestRegressor was not able to catch the relationship between the sea level, CO2 concentration and temperature, we have a constant temperature in the future. Now let's pass to another models

### Downloading the data and processing it again to reset it and making other transformation adapt to our models

In [ ]:
#If your lauching this code on  Jupyter Notebook you need to download  the temperature dataset from kaggle first the kaggle API encounter a problem with Jupyter on this dataset
file_path = "city_temperature.csv"
df_temperature = pad.read_csv(file_path)

file_path = "co2_conc.csv"
df_CO2_emission = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "arunavsutar/daily-atmosphere-carbon-dioxide-concentration",
        file_path,)

file_path = "sealevel.csv"
df_sea_level = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "kkhandekar/global-sea-level-1993-2021",
        file_path,)

In [ ]:
df_sea_level.drop(columns = ["TotalWeightedObservations","GMSL_noGIA","StdDevGMSL_noGIA","GMSL_GIA","StdDevGMSL_GIA","SmoothedGSML_GIA","SmoothedGSML_noGIA"])
df_sea_level = df_sea_level[(df_sea_level['Year'] >= 2013) & (df_sea_level['Year'] <= 2020)]
df_sea_level = df_sea_level.groupby(['Year']).mean().reset_index()

In [ ]:
df_temperature = df_temperature.drop(columns = ["Region","Country","State"])
df_temperature = df_temperature[(df_temperature['Year'] >= 2013) & (df_temperature['Year'] <= 2020)]
df_temperature  = df_temperature [df_temperature ['City'] == 'Paris']
df_temperature.rename(columns={'AvgTemperature': 'avg_city_temp'}, inplace=True)

In [ ]:
df_CO2_emission.drop(columns = ["Unnamed: 0","cycle"], inplace=True)
df_CO2_emission = df_CO2_emission[(df_CO2_emission['year'] >= 2013) & (df_CO2_emission['year'] <= 2020)]
df_CO2_emission.rename(columns={'year': 'Year'}, inplace=True)
df_CO2_emission.rename(columns={'month': 'Month'}, inplace=True)
df_CO2_emission.rename(columns={'day': 'Day'}, inplace=True)
df_CO2_emission.rename(columns={'trend': 'concentration_in_CO2'}, inplace=True)

In [ ]:
df_merged = df_temperature.merge(df_CO2_emission, on=['Year', 'Month', 'Day'], how='inner').merge(df_sea_level, on=['Year'], how='inner')

In [ ]:
df_paris = df_merged[df_merged['City'] == 'Paris']


In [ ]:
df_paris.head()

In [ ]:
df_paris_clean = df_paris[["Month","Day","Year","avg_city_temp","concentration_in_CO2"]]

In [ ]:
df_paris_clean.head()

### Testing LSTM

In [ ]:
df_paris_clean['avg_city_temp_C'] = (df_paris_clean['avg_city_temp'] - 32) * 5 / 9
df_paris_clean['date'] = pad.to_datetime(df_paris_clean[['Year', 'Month', 'Day']])
df_paris_clean = df_paris_clean.sort_values('date').reset_index(drop=True)

temps = df_paris_clean['avg_city_temp_C'].values.reshape(-1, 1)
scaler = MinMaxScaler()
temps_scaled = scaler.fit_transform(temps)

def create_sequences(data, window_size=30):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

window_size = 30
X, y = create_sequences(temps_scaled, window_size)

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = Sequential()
model.add(LSTM(64, activation='tanh', input_shape=(window_size, 1)))
model.add(Dense(32, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

y_pred_scaled = model.predict(X_test)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_inv = scaler.inverse_transform(y_pred_scaled).flatten()

mae = mean_absolute_error(y_test_inv, y_pred_inv)
rmse = mean_squared_error(y_test_inv, y_pred_inv, squared=False)
r2 = r2_score(y_test_inv, y_pred_inv)

last_sequence = temps_scaled[-window_size:].reshape(1, window_size, 1)
future_preds_scaled = []

for _ in range(5 * 365):
    pred_scaled = model.predict(last_sequence, verbose=0)[0, 0]
    future_preds_scaled.append(pred_scaled)
    last_sequence = np.append(last_sequence[:, 1:, :], [[[pred_scaled]]], axis=1)

future_preds = scaler.inverse_transform(np.array(future_preds_scaled).reshape(-1, 1)).flatten()

last_date = df_paris_clean['date'].iloc[-1]
dates_future = [last_date + pad.Timedelta(days=i+1) for i in range(len(future_preds))]
df_future = pad.DataFrame({'date': dates_future, 'avg_city_temp_C': future_preds})

df_historical = df_paris_clean[['date', 'avg_city_temp_C']]
df_full = pad.concat([df_historical, df_future])

df_full['year_month'] = df_full['date'].dt.to_period('M')
monthly_mean = df_full.groupby('year_month')['avg_city_temp_C'].mean().reset_index()
monthly_mean['year_month'] = monthly_mean['year_month'].dt.to_timestamp()

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

plt.figure(figsize=(14, 6))
plt.plot(monthly_mean['year_month'], monthly_mean['avg_city_temp_C'], marker='o')
plt.axvline(df_historical['date'].max(), color='red', linestyle='--', label="Début prédiction")
plt.title("Température moyenne mensuelle observée et prédite (LSTM avec MinMaxScaler)")
plt.xlabel("Mois")
plt.ylabel("Température moyenne mensuelle (°C)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

As we can see the LSTM model don't produce good results, denying the global warning with the temperature stagnates when he needs to predict new values

# Testing GRU

In [ ]:
df = df_paris_clean.copy()
df['avg_city_temp_C'] = (df['avg_city_temp'] - 32) * 5 / 9
df = df[(df['avg_city_temp_C'] <= 50) & (df['avg_city_temp_C'] >= -20)]
df['date'] = pad.to_datetime(df[['Year', 'Month', 'Day']])
df = df.sort_values('date')

df['year_month'] = df['date'].dt.to_period('M')
monthly = df.groupby('year_month').agg({
    'avg_city_temp_C': 'mean',
    'concentration_in_CO2': 'mean'
}).reset_index()
monthly['year_month'] = monthly['year_month'].dt.to_timestamp()

monthly['month'] = monthly['year_month'].dt.month
monthly['year'] = monthly['year_month'].dt.year

monthly['month_sin'] = np.sin(2 * np.pi * monthly['month'] / 12)
monthly['month_cos'] = np.cos(2 * np.pi * monthly['month'] / 12)

monthly['year_norm'] = (monthly['year'] - monthly['year'].min()) / (monthly['year'].max() - monthly['year'].min())

features = ['avg_city_temp_C', 'concentration_in_CO2', 'month_sin', 'month_cos', 'year_norm']
data = monthly[features].values

def create_sequences(data, seq_length=12):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length, :])
        y.append(data[i+seq_length, 0])
    return np.array(X), np.array(y)

SEQ_LENGTH = 12

X, y = create_sequences(data, SEQ_LENGTH)
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

#MinMaxScaler to avoids overfitting
scaler_X = MinMaxScaler()
X_train_reshaped = X_train.reshape(-1, X_train.shape[2])
X_train_scaled = scaler_X.fit_transform(X_train_reshaped).reshape(X_train.shape)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1,1)).flatten()

X_test_reshaped = X_test.reshape(-1, X_test.shape[2])
X_test_scaled = scaler_X.transform(X_test_reshaped).reshape(X_test.shape)
y_test_scaled = scaler_y.transform(y_test.reshape(-1,1)).flatten()

model = tf.keras.Sequential([
    tf.keras.layers.GRU(64, activation='relu', input_shape=(SEQ_LENGTH, len(features))),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')

history = model.fit(X_train_scaled, y_train_scaled, epochs=10, batch_size=6,
                    validation_data=(X_test_scaled, y_test_scaled))

y_pred_scaled = model.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

future_preds = []
last_seq = X_test_scaled[-1]

for i in range(60): # prediction on 60 months -> 5 years
    pred_scaled = model.predict(last_seq[np.newaxis, :, :])[0,0]
    future_preds.append(pred_scaled)
    last_seq_unscaled = scaler_X.inverse_transform(last_seq)
    last_co2 = last_seq_unscaled[-1, 1]
    co2_growth = (monthly['concentration_in_CO2'].diff().mean())
    new_co2 = last_co2 + co2_growth
    last_month = monthly['month'].iloc[-1] + i + 1
    new_month = ((last_month - 1) % 12) + 1
    new_year = monthly['year'].iloc[-1] + (last_month - 1) // 12
    year_min = monthly['year'].min()
    year_norm = (new_year - year_min) / (monthly['year'].max() - year_min)
    month_sin = np.sin(2 * np.pi * new_month / 12)
    month_cos = np.cos(2 * np.pi * new_month / 12)
    new_row_unscaled = np.array([scaler_y.inverse_transform([[pred_scaled]])[0,0], new_co2, month_sin, month_cos, year_norm])
    new_row_scaled = scaler_X.transform(new_row_unscaled.reshape(1,-1))[0]
    last_seq = np.vstack([last_seq[1:], new_row_scaled])

print(f"MAE : {mae:.3f}")
print(f"R2 score : {r2:.3f}")
future_preds_inv = scaler_y.inverse_transform(np.array(future_preds).reshape(-1,1)).flatten()
plt.figure(figsize=(14,6))
plt.plot(monthly['year_month'], monthly['avg_city_temp_C'], label='Historique', color='blue')
plt.plot(monthly['year_month'].iloc[split_idx + SEQ_LENGTH:], y_pred, label='Prédiction Test GRU multivarié', color='green')
future_dates = pad.date_range(start=monthly['year_month'].iloc[-1] + pad.DateOffset(months=1), periods=60, freq='MS')
plt.plot(future_dates, future_preds_inv, label='Prévision Future GRU (5 ans)', color='orange')
plt.axvline(monthly['year_month'].iloc[split_idx + SEQ_LENGTH], color='green', linestyle='--', label='Début Test')
plt.axvline(monthly['year_month'].max(), color='red', linestyle='--', label='Début Prévision')
plt.title('Prévision température mensuelle avec GRU multivarié (température + CO2 + saison + année)')
plt.xlabel('Date')
plt.ylabel('Température (°C)')
plt.legend()
plt.grid(True)
plt.show()

The GRU model produce good result as we can see with the metrics R2 score reaching 0.878. The graph confirm it as we can see the  green line, the predictions is approaching the blue line the real data.

### Reseting the data to apply new processing for other models

In [ ]:
#If your lauching this code on  Jupyter Notebook you need to download  the temperature dataset from kaggle first the kaggle API encounter a problem with Jupyter on this dataset
file_path = "city_temperature.csv" #The file path to your downloaded csv
df_temperature = pad.read_csv(file_path)

file_path = "co2_conc.csv" 
df_CO2_emission = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "arunavsutar/daily-atmosphere-carbon-dioxide-concentration",
        file_path,)

file_path = "sealevel.csv"
df_sea_level = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "kkhandekar/global-sea-level-1993-2021",
        file_path,)

In [ ]:
df_sea_level.drop(columns = ["TotalWeightedObservations","GMSL_noGIA","StdDevGMSL_noGIA","GMSL_GIA","StdDevGMSL_GIA","SmoothedGSML_GIA","SmoothedGSML_noGIA"])
df_sea_level = df_sea_level[(df_sea_level['Year'] >= 2013) & (df_sea_level['Year'] <= 2020)]
df_sea_level = df_sea_level.groupby(['Year']).mean().reset_index()

In [ ]:
df_temperature = df_temperature.drop(columns = ["Region","Country","State"])
df_temperature  = df_temperature [df_temperature ['City'] == 'Paris']
df_temperature.rename(columns={'AvgTemperature': 'avg_city_temp'}, inplace=True)

In [ ]:
df_CO2_emission.drop(columns = ["Unnamed: 0","cycle"], inplace=True)
df_CO2_emission = df_CO2_emission[(df_CO2_emission['year'] >= 2013) & (df_CO2_emission['year'] <= 2020)]
df_CO2_emission.rename(columns={'year': 'Year'}, inplace=True)
df_CO2_emission.rename(columns={'month': 'Month'}, inplace=True)
df_CO2_emission.rename(columns={'day': 'Day'}, inplace=True)
df_CO2_emission.rename(columns={'trend': 'concentration_in_CO2'}, inplace=True)

In [ ]:
df_merged = df_temperature.merge(df_CO2_emission, on=['Year', 'Month', 'Day'], how='inner').merge(df_sea_level, on=['Year'], how='inner')

In [ ]:
def predict_Prophet():

        df_training = df_merged[(df_merged["City"]== 'Paris')]
        df_training= df_training[df_training["avg_city_temp"] != -99]
        df_training["avg_city_temp"]=(df_training["avg_city_temp"]-32)*(5/9) #convert the temperature in Celsius

        df_training['ds'] = pad.to_datetime(df_training[['Year', 'Month', 'Day']])
        df_training.rename(columns={'avg_city_temp': 'y'}, inplace=True) #need to rename it for the model Prophet
        df_training = df_training[['ds', 'y']]
        
        df_training = df_training.sort_values('ds')

        # Split : 80% entraînement, 20% test
        train_size = int(len(df_training) * 0.8)
        train_df = df_training.iloc[:train_size]
        test_df = df_training.iloc[train_size:]

        model = Prophet()
        model.fit(train_df)
        
        future = test_df[['ds']]
        forecast = model.predict(future)
        
        y_true = test_df['y'].values
        y_pred = forecast['yhat'].values
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        
        print(f"MAE : {mae:.2f}")
        print(f"RMSE : {rmse:.2f}")
        print(f"R² : {r2:.2f}")
        

        future = model.make_future_dataframe(periods=365*30)
        forecast = model.predict(future)

        print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())
        
        fig = model.plot(forecast)
        plt.legend([
            'Prévision (yhat)', 
            'Incertitude basse (yhat_lower)', 
            'Incertitude haute (yhat_upper)', 
            'Observations'
        ], loc='upper left')
        plt.title("Prévision de la température sur 30 ans pour Paris")
        plt.xlabel("Date")
        plt.ylabel("Température")
        plt.show()

predict_Prophet()

The Prophet is adapted to the problem showing good results, but not as good as the GRU model.

In [ ]:
def predict_LGB():

        df_training = df_merged[['Year', 'Month', 'Day','concentration_in_CO2']]
        df_training = df_training.sort_values(['Year', 'Month', 'Day'])

        # Split : 80% entraînement, 20% test
        train_size = int(len(df_training) * 0.8)
        train_df = df_training.iloc[:train_size]
        test_df = df_training.iloc[train_size:]

        x_train = train_df.drop(["concentration_in_CO2"],axis=1).to_numpy()
        y_train = train_df["concentration_in_CO2"].to_numpy()

        x_test = test_df.drop(["concentration_in_CO2"],axis=1).to_numpy()
        y_test = test_df["concentration_in_CO2"].to_numpy()

        params = {
            'num_leaves': 55,
            'learning_rate': 0.4
        }

        model = lgb.LGBMRegressor(**params)
        model.fit(x_train,y_train)

        y_pred = model.predict(x_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        print(f"MAE: {mae}")
        print(f"RMSE: {rmse}")

predict_LGB()  

LGB produce good results on the metrics, we hadn't the time to produce a graphic vue of this result